# Tasks 6 & 7: Feature Selection and Predictive Model

This notebook defines the features for predicting total piece travel time, trains an XGBoost Regressor, and evaluates its performance.

### The prediction problem

**Goal**: Predict the total travel time to the quench bath (`lifetime_bath_s`) using only data available **early in the process** — before the piece has finished the line.

**Why this is useful**: If we can predict the total time after only the 2nd strike, we can raise real-time alerts for pieces that are likely to be slow, allowing operators to investigate the cause while the piece is still on the line.

### Feature selection rationale

**Constraint**: the model must predict using only information available **after the 2nd strike** — approximately 18 seconds into the ~58-second journey. Any feature that requires waiting for later stages cannot be used, because by the time those values exist, the prediction is no longer useful.

#### Selected features

| Feature | Type | Available at | Why include it |
|---|---|---|---|
| `die_matrix` | Categorical (int) | Before processing | Each die has different tooling geometry and expected times — matrix 4974 has a median bath time of ~56s while 5091 has ~59s |
| `lifetime_2nd_strike_s` | Continuous (seconds) | After 2nd strike (~18s) | The earliest cumulative time — if the piece is already slow here, it will likely carry that delay through to the bath |
| `oee_cycle_time_s` | Continuous (seconds) | Rolling metric | Production rate context — slower OEE may correlate with systematic delays (robot trajectory adjustments, hydraulic pressure drops) |

#### Excluded features

| Feature | Why excluded |
|---|---|
| `lifetime_3rd_strike_s` | Available too late (~25s) — the piece is already halfway through the main press |
| `lifetime_4th_strike_s` | Available too late (~38s) — and has ~16% missing data from a sensor offline period |
| `lifetime_auxiliary_press_s` | Available too late (~55s) — only ~2s before the bath, prediction would be useless |
| `lifetime_general_s` | Equivalent to `lifetime_bath_s` — redundant with the target |
| `partial_*` columns | Derived from cumulative times that include late-stage data |
| `piece_id` | Not predictive — just an identifier |

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

gold_path = Path("../data/gold/pieces.parquet")
if not gold_path.exists():
    raise FileNotFoundError(f"Gold parquet not found: {gold_path.resolve()}")

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET_COL = "lifetime_bath_s"

print("Gold path:", gold_path.resolve())
print("Models dir:", models_dir.resolve())


Gold path: C:\Users\warre\VaultTech\data\gold\pieces.parquet
Models dir: C:\Users\warre\VaultTech\models


## 1. Load gold dataset

In [3]:
df = pd.read_parquet(gold_path)
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head(10))


Dataset shape: (140404, 18)
Columns: ['timestamp', 'piece_id', 'die_matrix', 'lifetime_2nd_strike_s', 'lifetime_3rd_strike_s', 'lifetime_4th_strike_s', 'lifetime_auxiliary_press_s', 'lifetime_bath_s', 'lifetime_general_s', 'partial_furnace_to_2nd_strike_s', 'partial_2nd_to_3rd_strike_s', 'partial_3rd_to_4th_strike_s', 'partial_4th_strike_to_auxiliary_press_s', 'partial_auxiliary_press_to_bath_s', 'oee_cycle_time_s', 'seconds_since_prev_piece', 'is_production_gap', 'production_run_id']


,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s,lifetime_general_s,partial_furnace_to_2nd_strike_s,partial_2nd_to_3rd_strike_s,partial_3rd_to_4th_strike_s,partial_4th_strike_to_auxiliary_press_s,partial_auxiliary_press_to_bath_s,oee_cycle_time_s,seconds_since_prev_piece,is_production_gap,production_run_id
0,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,24.600000,38.000000,54.599998,56.200001,56.200001,17.900000,6.700001,13.400000,16.599998,1.600002,NaN,NaN,False,1
1,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,24.600000,37.900002,54.799999,56.400002,56.400002,17.900000,6.700001,13.300001,16.899998,1.600002,NaN,12.708,False,1
2,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,24.799999,38.299999,55.299999,56.900002,56.900002,18.200001,6.599998,13.500000,17.000000,1.600002,NaN,14.609,False,1
3,2025-11-06 15:25:56.462000+00:00,251106001724,5052,18.400000,25.100000,38.400002,55.500000,57.099998,57.099998,18.400000,6.700001,13.300001,17.099998,1.599998,NaN,12.719,False,1
4,2025-11-06 15:26:10.462000+00:00,251106001725,5052,18.200001,24.799999,38.200001,55.500000,57.200001,57.200001,18.200001,6.599998,13.400002,17.299999,1.700001,NaN,14.000,False,1
5,2025-11-06 15:26:23.771000+00:00,251106001726,5052,16.700001,23.400000,36.799999,53.599998,55.200001,55.200001,16.700001,6.699999,13.400000,16.799999,1.600002,13.4690,13.309,False,1
6,2025-11-06 15:26:36.382000+00:00,251106001727,5052,18.299999,24.900000,38.200001,54.700001,56.400002,56.400002,18.299999,6.600000,13.300001,16.500000,1.700001,13.4496,12.611,False,1
7,2025-11-06 15:26:50.095000+00:00,251106001728,5052,17.900000,24.500000,37.700001,54.700001,56.299999,56.299999,17.900000,6.600000,13.200001,17.000000,1.599998,13.2704,13.713,False,1
8,2025-11-06 15:27:57.427000+00:00,251106001733,5052,16.700001,23.299999,36.599998,53.299999,54.900002,54.900002,16.700001,6.599998,13.299999,16.700001,1.600002,NaN,67.332,False,1
9,2025-11-06 15:29:04.470000+00:00,251106001738,5052,16.700001,23.600000,36.799999,54.099998,55.799999,55.799999,16.700001,6.900000,13.199999,17.299999,1.700001,NaN,67.043,False,1


## 2. Prepare features and target

- **Features (X)**: `die_matrix`, `lifetime_2nd_strike_s`, `oee_cycle_time_s`
- **Target (y)**: `lifetime_bath_s`

Drop rows where any feature or the target is NULL (missing drill data does not affect us here since we don't use drill as a feature).

In [4]:
selected_features = [
    "die_matrix",
    "lifetime_2nd_strike_s",
    "oee_cycle_time_s",
]

excluded_features = [
    "lifetime_3rd_strike_s",
    "lifetime_4th_strike_s",
    "lifetime_auxiliary_press_s",
    "lifetime_general_s",
    "lifetime_bath_s",
    "piece_id",
]

print("Selected features:", selected_features)
print("Target:", TARGET_COL)
print("Excluded features:", excluded_features)

model_df = df[selected_features + [TARGET_COL]].copy()

# Fill missing OEE with global median, per design doc
oee_median = model_df["oee_cycle_time_s"].median()
model_df["oee_cycle_time_s"] = model_df["oee_cycle_time_s"].fillna(oee_median)

# Drop rows missing required modeling fields
model_df = model_df.dropna(subset=["die_matrix", "lifetime_2nd_strike_s", TARGET_COL]).copy()

print("Model dataset shape after NA handling:", model_df.shape)
display(model_df.head(10))

Selected features: ['die_matrix', 'lifetime_2nd_strike_s', 'oee_cycle_time_s']
Target: lifetime_bath_s
Excluded features: ['lifetime_3rd_strike_s', 'lifetime_4th_strike_s', 'lifetime_auxiliary_press_s', 'lifetime_general_s', 'lifetime_bath_s', 'piece_id']
Model dataset shape after NA handling: (140404, 4)


,die_matrix,lifetime_2nd_strike_s,oee_cycle_time_s,lifetime_bath_s
0,5052,17.900000,13.7098,56.200001
1,5052,17.900000,13.7098,56.400002
2,5052,18.200001,13.7098,56.900002
3,5052,18.400000,13.7098,57.099998
4,5052,18.200001,13.7098,57.200001
5,5052,16.700001,13.4690,55.200001
6,5052,18.299999,13.4496,56.400002
7,5052,17.900000,13.2704,56.299999
8,5052,16.700001,13.7098,54.900002
9,5052,16.700001,13.7098,55.799999


## 3. Feature correlation with target

How strongly does each feature correlate with the total bath time? High correlation suggests predictive value.

In [5]:
numeric_features = [
    "lifetime_2nd_strike_s",
    "oee_cycle_time_s",
]

correlation_summary = pd.DataFrame({
    "feature": numeric_features,
    "pearson_corr_with_target": [
        model_df[feature].corr(model_df[TARGET_COL]) for feature in numeric_features
    ],
})

correlation_summary["pearson_corr_with_target"] = correlation_summary["pearson_corr_with_target"].round(4)

display(correlation_summary)


,feature,pearson_corr_with_target
0,lifetime_2nd_strike_s,0.7574
1,oee_cycle_time_s,0.2807


## 4. Train/test split

Split 80/20 with a fixed random seed for reproducibility. Stratify by die_matrix to ensure each matrix is represented proportionally in both sets.

In [7]:
X = model_df[selected_features].copy()
y = model_df[TARGET_COL].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=model_df["die_matrix"],
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

print("Train die_matrix distribution:")
display(X_train["die_matrix"].value_counts(normalize=True).sort_index().round(4))

print("Test die_matrix distribution:")
display(X_test["die_matrix"].value_counts(normalize=True).sort_index().round(4))


Train shape: (112323, 3) (112323,)
Test shape: (28081, 3) (28081,)
Train die_matrix distribution:


die_matrix
4974    0.1106
5052    0.1488
5090    0.5817
5091    0.1589
Name: proportion, dtype: float64

Test die_matrix distribution:


die_matrix
4974    0.1106
5052    0.1487
5090    0.5817
5091    0.1589
Name: proportion, dtype: float64

## 5. Train XGBoost Regressor

XGBoost is chosen because:
- Handles mixed feature types (categorical die_matrix + continuous times)
- Robust to the remaining noise in the data
- Fast training on ~100k rows
- Produces feature importance rankings

In [9]:
model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    objective="reg:squarederror",
)

model.fit(X_train, y_train)

print("Model trained successfully.")


Model trained successfully.


## 6. Evaluate on test set

Key metrics:
- **RMSE**: root mean squared error (same unit as target — seconds)
- **MAE**: mean absolute error (average prediction error in seconds)
- **R²**: coefficient of determination (1.0 = perfect, 0.0 = no better than mean)

In [10]:
y_pred = model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

metrics_summary = pd.DataFrame([{
    "rmse_s": round(rmse, 4),
    "mae_s": round(mae, 4),
    "r2": round(r2, 4),
}])

display(metrics_summary)

print(f"RMSE: {rmse:.4f} s")
print(f"MAE:  {mae:.4f} s")
print(f"R²:   {r2:.4f}")

,rmse_s,mae_s,r2
0,1.8546,0.937,0.6622


RMSE: 1.8546 s
MAE:  0.9370 s
R²:   0.6622


## 7. Performance per die matrix

Check if the model performs equally well across all matrices, or if some are harder to predict.

In [11]:
test_results = X_test.copy()
test_results["y_true"] = y_test.values
test_results["y_pred"] = y_pred

per_matrix_metrics = []

for matrix, group in test_results.groupby("die_matrix"):
    rmse_m = mean_squared_error(group["y_true"], group["y_pred"]) ** 0.5
    mae_m = mean_absolute_error(group["y_true"], group["y_pred"])
    r2_m = r2_score(group["y_true"], group["y_pred"])

    per_matrix_metrics.append({
        "die_matrix": matrix,
        "n_samples": len(group),
        "rmse_s": round(rmse_m, 4),
        "mae_s": round(mae_m, 4),
        "r2": round(r2_m, 4),
    })

per_matrix_metrics = (
    pd.DataFrame(per_matrix_metrics)
    .sort_values("die_matrix")
    .reset_index(drop=True)
)

display(per_matrix_metrics)

,die_matrix,n_samples,rmse_s,mae_s,r2
0,4974,3106,0.8966,0.4631,0.7427
1,5052,4177,1.5146,0.7669,0.6875
2,5090,16336,2.0746,1.1219,0.6260
3,5091,4462,1.7841,0.7487,0.6660


## 8. Feature importance

Which features contribute most to the prediction? This validates the feature selection rationale.

In [12]:
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_,
})

feature_importance = feature_importance.sort_values(
    "importance", ascending=False
).reset_index(drop=True)

display(feature_importance)


,feature,importance
0,lifetime_2nd_strike_s,0.746212
1,oee_cycle_time_s,0.177497
2,die_matrix,0.076291


## 9. Save model and metadata

Save the trained model and its metadata for use by the inference service (Task 8).

In [13]:
from pathlib import Path
import json

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "xgboost_bath_predictor.json"
metadata_path = models_dir / "model_metadata.json"

# Save model
model.save_model(model_path)

# Save metadata
metadata = {
    "model_type": "XGBoostRegressor",
    "target": TARGET_COL,
    "features": selected_features,
    "fillna_strategy": {
        "oee_cycle_time_s": {
            "method": "median",
            "value": float(oee_median),
        }
    },
    "train_test_split": {
        "test_size": 0.2,
        "random_state": RANDOM_STATE,
        "stratified_by": "die_matrix",
    },
    "hyperparameters": {
        "n_estimators": 200,
        "max_depth": 6,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "objective": "reg:squarederror",
    },
    "metrics_overall": {
        "rmse_s": float(rmse),
        "mae_s": float(mae),
        "r2": float(r2),
    },
    "metrics_per_matrix": per_matrix_metrics.to_dict(orient="records"),
    "feature_importance": feature_importance.to_dict(orient="records"),
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Model saved to:", model_path.resolve())
print("Metadata saved to:", metadata_path.resolve())


Model saved to: C:\Users\warre\VaultTech\models\xgboost_bath_predictor.json
Metadata saved to: C:\Users\warre\VaultTech\models\model_metadata.json


## 10. Summary

### Feature importance

Feature importance confirms and strengthens the feature selection rationale:

- **`lifetime_2nd_strike_s` (~74.6%)** is by far the dominant predictor  
  - This shows that early-stage delays strongly propagate through the process  
  - A piece that is already slow at the 2nd strike is very likely to remain slow until the bath  

- **`oee_cycle_time_s` (~17.7%)** provides meaningful secondary signal  
  - It captures the overall state of the line (throughput, congestion, minor slowdowns)  
  - While weaker than the direct timing signal, it improves predictions when combined with it  

- **`die_matrix` (~7.6%)** contributes less directly but remains essential  
  - It encodes matrix-specific baselines and process differences  
  - Its relatively low importance suggests that most variation is driven by piece-level timing rather than matrix identity alone  

Overall, the model relies primarily on **early observed delay (`lifetime_2nd_strike_s`)**, with contextual adjustments from **line conditions (`oee_cycle_time_s`)** and **matrix-specific baselines (`die_matrix`)**.

This validates the modeling approach: early-stage timing contains the strongest predictive signal for final travel time.
